## Data Understanding (Ententimento dos dados)

### Biblioteca / Configuração

In [ ]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

In [ ]:
# Acesso aos modulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

# Manipulação dos dados
import pandas as pd
import numpy as np

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Diretórios
from config.paths import *
from config.function_basic import *

# Avisos
import warnings
warnings.filterwarnings('ignore')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('Ambiente Configurado')

### Carregamento dos dados 

In [ ]:
# lê o arquivo parquet e carrega em um DataFrame
abt00 = pd.read_parquet(RAW_DIR / "book_variaveis_04_v2.parquet")

### Visualizando os dados

In [ ]:
# Visualizando os dados 
print('Visualizando os dados'.upper())
print('=' * 30)
show_dataframe_samples(abt00)

#### Informações Basicas

In [ ]:
print('imformações básicas'.upper())
print('=' * 30)

basic_information(abt00)

In [ ]:
print('imformações gerais'.upper())
print('=' * 30)

info_table(abt00)

### Verificar colunas inf

In [ ]:
# Para análise dos dados força colunas numéricas para int e o resto para object, e detecta colunas com inf
cols_com_inf = []

# colunas fora do casting automático (IDs, datas, chaves)
ignore_cols = ['NUM_CPF', 'DATADENASCIMENTO', 'DATA_SAFRA']

for col in abt00.columns:

    # ignora colunas fora do escopo
    if col in ignore_cols:
        continue

    coerced = pd.to_numeric(abt00[col], errors='coerce')

    if coerced.notna().any():
        # detecta infinito
        if np.isinf(coerced).any():
            cols_com_inf.append(col)

        coerced = coerced.replace([np.inf, -np.inf], pd.NA)

        # decide tipo
        if (coerced.dropna() % 1 != 0).any():
            abt00[col] = coerced.astype('Float64')
        else:
            abt00[col] = coerced.astype('Int64')
    else:
        abt00[col] = abt00[col].astype(str)

cols_com_inf

#### Tipo dos dados

In [ ]:
# Tipo dos dados
print('tipo dos dados'.upper())
print('=' * 30)

data_type(abt00)

## Qualidade dos dados

#### Duplicados

In [ ]:
# Verificação de duplicatas
print('análise de duplicatas'.upper())
print('=' * 30)

check_duplicates(abt00)

In [ ]:
# encontrar linhas duplicadas considerando CPF + SAFRA
duplicados = (
    abt00[abt00.duplicated(subset=['NUM_CPF', 'SAFRA'], keep=False)]
    .sort_values(['NUM_CPF', 'SAFRA'])
)

if not duplicados.empty:
    print("Tem duplicados (CPF + SAFRA)")
    display(duplicados.head(5))
else:
    print("Sem duplicados")

#### Valores Faltantes

In [ ]:
print('análise de valores faltantes'.upper())
print('=' * 30)

analyze_missing_values(abt00)

## Análise dos dados

In [ ]:
# define colunas a serem ignoradas na separação
ignore_cols = ['SAFRA', 'FPD', 'NUM_CPF',]

# Separando os dados por tipo de colunas
numericos = abt00.select_dtypes(include=['int64','Int64', 'float64']).drop(columns=ignore_cols, errors='ignore')
categoricos = abt00.select_dtypes(include=['object']).drop(columns=ignore_cols, errors='ignore')

### Variáveis categóricas

In [ ]:
# Exploração de variáveis categóricas
print('variáveis categóricas'.upper())
print('=' * 30)

analyze_categorical_features(categoricos)

### Variáveis numéricas

In [ ]:
### Variáveis numéricas
print('variáveis numéricas'.upper())
print('=' * 30)

analyze_numerical_features(numericos)

In [ ]:
# Visualização exploratória de distribuição e outliers para variáveis numéricas
#plot_numerical_dual_charts(abt00, numericos)